# 01 — Ambiente de aprendizado por reforço

Este notebook implementa a simulação de duelos do Deadlock com cinco heróis, 13 itens e 20 habilidades. As informações são lidas de quatro arquivos CSV.

**Como executar no Colab:** envie os quatro CSVs ou `dados_ambiente_balanceado.zip` para a área Arquivos e execute as células em ordem. Se necessário, instale `pandas`, `numpy` e `numba` com `%pip install pandas numpy numba`.

Os dados são entradas fixas do ambiente. A alteração dos atributos ou das regras exige verificar a compatibilidade do modelo.

## 1. Disponibilizar os CSVs

In [ ]:
from pathlib import Path
import zipfile

# Os quatro CSVs descrevem os heróis, itens, armas e habilidades.
ARQUIVOS_CSV = ["dataset_heroes.csv", "dataset_itens_compraveis.csv",
                "alcance_armas.csv", "habilidades_simuladas.csv"]
RAIZ = Path.cwd()

# Se os arquivos estiverem compactados, extrai apenas as tabelas necessárias.
zip_dados = RAIZ / "dados_ambiente_balanceado.zip"
if zip_dados.is_file():
    with zipfile.ZipFile(zip_dados) as pacote:
        for nome in ARQUIVOS_CSV:
            destino = RAIZ / nome
            if not destino.is_file():
                destino.write_bytes(pacote.read(nome))

# Verifica se todos os arquivos estão disponíveis antes de chamar o Pandas.
faltam = [nome for nome in ARQUIVOS_CSV if not (RAIZ / nome).is_file()]
if faltam:
    raise FileNotFoundError("Envie os CSVs: " + ", ".join(faltam))
print("CSV disponíveis:", ", ".join(ARQUIVOS_CSV))

## 2. Configurações e leitura dos dados

In [ ]:
"""Deadlock: ambiente simplificado de compras e duelos entre heróis."""
import hashlib  # Identifica os dados e o código usados pelo modelo.
import json  # Interpreta as propriedades dentro do CSV, sem exportar JSON.
import math  # Valida parâmetros numéricos.
import random  # Sorteia os adversários.
from functools import lru_cache  # Reutiliza cálculos determinísticos.
from pathlib import Path  # Localiza os arquivos do projeto.

import pandas as pd  # Lê e organiza as tabelas da coleta.

# Path.cwd() indica a pasta de trabalho atual, normalmente /content no Colab.
RAIZ = Path.cwd()
DADOS = RAIZ
# Identificador usado para verificar a compatibilidade entre ambiente e modelo.
CODIGO_SHA256 = globals().get("CODIGO_SHA256", "d86db854c14488449e0dd946058363786db14c830cb3c59a897bb6a2ce932419")
VERSAO_AMBIENTE = "6.0-acertos-recompensa"
# A acao -1 representa nao comprar nenhum item; as outras acoes usam o ID do item.
ACAO_GUARDAR = -1
# Parametros do episodio: no maximo quatro duelos e 800 almas por vitoria.
ETAPAS = 4
ALMAS_INICIAIS = 800
ALMAS_POR_VITORIA = 800
# Os saldos e percentuais de vida sao discretos para limitar a quantidade de estados.
SALDOS = (0, 800, 1600, 2400, 3200)
VIDAS_PCT = (25, 50, 75, 100)
# O limite de dois itens vale para cenarios iniciais; novas compras podem ampliar o inventario.
MAX_ITENS_INICIAIS = 2
# Recuperacao entre duelos em segundos. DT e o tamanho de cada passo da simulacao.
TEMPO_RECUPERACAO = 45
DT = 0.1
TEMPO_MAX_COMBATE = 120
FRACAO_VIDA_INIMIGO = 1.0  # 1.0 = vida cheia; altere e retreine para outro cenário.

# Distância fixa por duelo, sorteada a cada etapa; não há deslocamento.
DISTANCIAS = (2, 15, 25)
HEROIS = ['Infernus', 'Viscoso', 'Vindicta', 'Abrams', 'Guarda']

# As tabelas preparadas são entradas fixas do simulador (não há coleta pela API).
# Cada habilidade registra as propriedades originais e as aproximações utilizadas.
# Lista de atributos numericos que precisam ser finitos e nao negativos nas habilidades.
campos_habilidade = ["dano", "escala_dano", "cura", "escala_cura", "roubo_vida",
                     "alcance_m", "duracao_s", "atraso_s", "cooldown_s",
                     "acumulo_por_bala", "max_acumulos", "janela_acumulo_s"]
# pd.read_csv cria um DataFrame; utf-8-sig aceita a marca BOM de arquivos UTF-8.
tabela_habilidades = pd.read_csv(DADOS / "habilidades_simuladas.csv", encoding="utf-8-sig")
tabela_alcances = pd.read_csv(DADOS / "alcance_armas.csv", encoding="utf-8-sig")
# Valida os cinco herois, evita duplicatas de habilidade e de alcance e verifica os numeros.
if (set(tabela_habilidades.heroi) != set(HEROIS) or
        tabela_habilidades.duplicated(["heroi", "slot"]).any() or
        not tabela_habilidades[campos_habilidade].map(lambda x: math.isfinite(float(x)) and x >= 0).all().all() or
        set(tabela_alcances.heroi) != set(HEROIS) or tabela_alcances.heroi.duplicated().any()):
    raise ValueError("Tabelas de alcance ou habilidades inválidas.")
# Agrupa as habilidades em uma lista de dicionarios por heroi e indexa os alcances pelo nome.
habilidades_base = {h: tabela_habilidades.loc[tabela_habilidades.heroi == h].to_dict("records") for h in HEROIS}
alcances_armas = tabela_alcances.set_index("heroi").to_dict("index")

# Seleção de 13 itens passivos; sem componentes ou compras repetidas.
# Cada efeito declara a propriedade exata do catálogo que será usada.
# Condições: 'sempre', 'acima65' e 'abaixo50'.
# Percentuais são divididos por 100, valores absolutos não.
# Cada tupla liga uma propriedade do CSV ao atributo do simulador, divisor e condicao de ativacao.
REGRAS_ITENS = {
    "upgrade_health": [("BonusHealth", "vida_maxima", 1, "sempre")],
    "upgrade_endurance": [("BonusHealthRegen", "regeneracao", 1, "sempre"),
                          ("OutOfCombatHealthRegen", "regeneracao_fora_combate", 1, "sempre")],
    "upgrade_improved_spirit": [("TechPower", "poder_espiritual", 1, "sempre")],
    "upgrade_rapid_rounds": [("BonusFireRate", "bonus_cadencia", 100, "sempre")],
    "upgrade_clip_size": [("BaseAttackDamagePercent", "bonus_dano_arma", 100, "sempre"),
                          ("BonusClipSizePercent", "bonus_carregador", 100, "sempre")],
    "upgrade_regenerating_bullet_shield": [("BulletResist", "resistencia_balistica", 100, "sempre"),
        ("OutOfCombatHealthRegen", "regeneracao_fora_combate", 1, "sempre"),
        ("BaseAttackDamagePercent", "bonus_dano_arma", 100, "acima65"),
        ("BonusFireRate", "bonus_cadencia", 100, "acima65")],
    "upgrade_magic_shield": [("TechResist", "resistencia_espiritual", 100, "sempre"),
        ("OutOfCombatHealthRegen", "regeneracao_fora_combate", 1, "sempre"),
        ("TechPower", "poder_espiritual", 1, "acima65"),
        ("CooldownReduction", "reducao_cooldown", 100, "acima65")],
    # Itens de roubo de vida, cura, recarga e defesa.
    "upgrade_vampire": [("BonusHealth", "vida_maxima", 1, "sempre"),
        ("BaseAttackDamagePercent", "bonus_dano_arma", 100, "sempre"),
        ("BulletLifestealPercent", "roubo_vida_balistico", 100, "sempre")],
    "upgrade_health_stealing_magic": [("BonusHealth", "vida_maxima", 1, "sempre"),
        ("TechPower", "poder_espiritual", 1, "sempre"),
        ("AbilityLifestealPercentHero", "roubo_vida_espiritual", 100, "sempre")],
    "upgrade_healing_booster": [("OutOfCombatHealthRegen", "regeneracao_fora_combate", 1, "sempre"),
        ("HealAmpCastPercent", "amplificacao_cura", 100, "sempre"),
        ("HealAmpRegenPercent", "amplificacao_regeneracao", 100, "sempre")],
    # Compressão de Recarga é aplicada somente à primeira habilidade ativa simulada.
    "upgrade_magic_tempo": [("CooldownReduction", "reducao_cooldown_infusao", 100, "sempre")],
    "upgrade_improved_bullet_armor": [("BulletResist", "resistencia_balistica", 100, "sempre"),
        ("OutOfCombatHealthRegen", "regeneracao_fora_combate", 1, "sempre"),
        ("BulletResistBelowThreshold", "resistencia_balistica", 100, "abaixo50")],
    "upgrade_tech_purge": [("TechResist", "resistencia_espiritual", 100, "sempre"),
        ("OutOfCombatHealthRegen", "regeneracao_fora_combate", 1, "sempre"),
        ("TechResistBelowThreshold", "resistencia_espiritual", 100, "abaixo50")],
}


# Le os dois CSVs principais e seleciona os registros que realmente fazem parte do ambiente.
def carregar_dados():
    """Seleciona os heróis e itens pelos identificadores do catálogo."""
    tabela_h = pd.read_csv(DADOS / "dataset_heroes.csv", encoding="utf-8-sig")
    # Le o catalogo inteiro; os efeitos dos itens estao na coluna properties_json.
    tabela_i = pd.read_csv(DADOS / "dataset_itens_compraveis.csv", encoding="utf-8-sig")
    # set_index e loc selecionam os cinco herois pelo nome; reset_index restaura a coluna.
    h = tabela_h.set_index("nome_heroi").loc[HEROIS].reset_index()
    # Seleciona os 13 itens pelo class_name, identificador interno do catalogo, nao pela traducao.
    i = tabela_i.set_index("class_name").loc[list(REGRAS_ITENS)].reset_index()
    campos_h = ["vida_base", "regeneracao_base", "damage_per_shot", "shots_per_second",
                "clip_size", "reload_duration"]
    # Confere IDs unicos e atributos numericos para impedir simulacoes com dados incorretos.
    if len(h) != len(HEROIS) or h["hero_id"].duplicated().any():
        raise ValueError("Heróis duplicados na coleta.")
    if not h[campos_h].map(lambda x: math.isfinite(float(x))).all().all():
        raise ValueError("Há atributos de heróis ausentes ou inválidos.")
    if (h[["vida_base", "damage_per_shot", "shots_per_second", "clip_size", "reload_duration"]] <= 0).any().any():
        raise ValueError("Atributos de arma ou vida inválidos.")
    # Aceita apenas os itens passivos compraveis com custo positivo.
    if (len(i) != len(REGRAS_ITENS) or i["item_id"].duplicated().any()
            or not i["shopable"].eq(True).all() or not i["activation"].eq("passive").all()
            or i["custo"].isna().any() or (i["custo"] <= 0).any()):
        raise ValueError("Itens selecionados inválidos no catálogo.")
    # Monta um dicionario: item_id -> lista de efeitos aplicados ao combatente.
    efeitos = {}
    for linha in i.itertuples():
        # json.loads converte o texto JSON guardado na celula CSV em um dicionario Python.
        props = json.loads(linha.properties_json)
        regras = []
        for prop, atributo, divisor, condicao in REGRAS_ITENS[linha.class_name]:
            # Cada regra busca o campo value da propriedade; divide percentuais por 100.
            valor = float(props[prop]["value"]) / divisor
            if not math.isfinite(valor):
                raise ValueError(f"Efeito inválido: {linha.nome}/{prop}")
            # Guarda o atributo, a magnitude e o momento em que o bonus deve ser aplicado.
            regras.append((atributo, valor, condicao))
        # Confere os limiares exigidos pelas condições de ativação.
        for chave, esperado in [("LifeThreshold", 65), ("HealthThreshold", 50)]:
            if chave in props and any(c == ("acima65" if esperado == 65 else "abaixo50") for _, _, c in regras):
                if float(props[chave]["value"]) != esperado:
                    raise ValueError(f"Limiar alterado em {linha.nome}; revise as regras.")
        efeitos[int(linha.item_id)] = regras
    return h, i, efeitos


# Transforma as tabelas filtradas em estruturas que o simulador consulta rapidamente.
heroes, itens, efeitos_itens = carregar_dados()
config_herois = heroes.set_index("nome_heroi").to_dict(orient="index")
# Mapeamentos entre nome, ID e custo facilitam compras e mensagens do chatbot.
ids_itens = dict(zip(itens["nome"], itens["item_id"].astype(int)))
nomes_itens = {int(r.item_id): r.nome for r in itens.itertuples()}
custos_itens = {int(r.item_id): int(r.custo) for r in itens.itertuples()}
adversarios = {heroi: config_herois[heroi] for heroi in HEROIS}


# SHA-256 funciona como uma impressao digital dos dados; detecta mudancas de versao.
def assinatura_dados():
    return {nome: hashlib.sha256((DADOS / nome).read_text(encoding="utf-8").encode("utf-8")).hexdigest()
            for nome in ("dataset_heroes.csv", "dataset_itens_compraveis.csv", "alcance_armas.csv", "habilidades_simuladas.csv")}


# Registra a configuração do ambiente para validar o modelo treinado.
def configuracao_ambiente():
    return {"versao": VERSAO_AMBIENTE, "etapas": ETAPAS,
            "almas_iniciais": ALMAS_INICIAIS, "almas_por_vitoria": ALMAS_POR_VITORIA,
            "tempo_recuperacao": TEMPO_RECUPERACAO, "dt": DT,
            "saldos": SALDOS, "vidas_pct": VIDAS_PCT, "max_itens_iniciais": MAX_ITENS_INICIAIS,
            "distancias": DISTANCIAS, "tempo_max_combate": TEMPO_MAX_COMBATE, "fracao_vida_inimigo": FRACAO_VIDA_INIMIGO,
            "codigo_sha256": CODIGO_SHA256}


def nome_acao(acao):
    return "Guardar almas" if acao == ACAO_GUARDAR else nomes_itens[acao]

## 3. Compras e atributos dos heróis

In [ ]:
# Uma acao valida e guardar almas ou comprar um item acessivel que ainda nao se possui.
def acoes_validas(estado):
    if estado.get("terminado", False) or estado["etapas_restantes"] <= 0 or estado["vida_atual"] <= 0:
        return []
    return [ACAO_GUARDAR] + [a for a, custo in custos_itens.items()
                           if custo <= estado["almas"] and a not in estado["inventario"]]


# Copia o estado antes de alterar saldo e inventario, preservando o estado anterior.
def executar_acao(estado, acao):
    if acao not in acoes_validas(estado):
        raise ValueError("Ação inválida para o estado atual.")
    novo = {**estado, "inventario": list(estado["inventario"])}
    if acao != ACAO_GUARDAR:
        novo["almas"] -= custos_itens[acao]
        novo["inventario"].append(int(acao))
    return novo


# lru_cache reaproveita atributos ja calculados para o mesmo heroi, inventario e faixa de vida.
@lru_cache(maxsize=30000)
def _atributos(heroi, inventario, acima65, abaixo50):
    # Comeca com atributos do heroi e soma regeneracao passiva de habilidades de resiliencia.
    base = config_herois[heroi]
    a = {"vida_maxima": float(base["vida_base"]),
         "regeneracao": float(base["regeneracao_base"]) + sum(h["cura"] for h in habilidades_base[heroi] if h["tipo"] == "resiliencia"), "regeneracao_fora_combate": 0.0,
         "poder_espiritual": 0.0, "bonus_dano_arma": 0.0, "bonus_cadencia": 0.0,
         "bonus_carregador": 0.0, "resistencia_balistica": 0.0,
         "resistencia_espiritual": 0.0, "reducao_cooldown": 0.0, "reducao_cooldown_infusao": 0.0,
         "roubo_vida_balistico": 0.0, "roubo_vida_espiritual": 0.0,
         "amplificacao_cura": 0.0, "amplificacao_regeneracao": 0.0}
    # Aplica os bonus de todos os itens conforme a proporcao atual de vida do heroi.
    for item_id in inventario:
        for atributo, valor, condicao in efeitos_itens[item_id]:
            if condicao == "sempre" or (condicao == "acima65" and acima65) or (condicao == "abaixo50" and abaixo50):
                a[atributo] += valor
    # Acumulação aditiva e limites: regras explícitas deste simulador.
    for k in ("resistencia_balistica", "resistencia_espiritual", "reducao_cooldown"):
        a[k] = min(0.90, a[k])
    return a


# Converte a vida em duas condicoes booleanas para ativar os bonus condicionais.
def calcular_atributos(inventario, proporcao_vida=1, heroi="Infernus"):
    return _atributos(heroi, tuple(sorted(inventario)), proporcao_vida > .65, proporcao_vida < .50).copy()


# Ao aumentar a vida maxima, conserva a mesma proporcao de vida atual apos a compra.
def atualizar_atributos(estado):
    proporcao = estado["vida_atual"] / estado["vida_maxima"]
    atributos = calcular_atributos(estado["inventario"], proporcao, estado["heroi"])
    estado["vida_maxima"] = atributos["vida_maxima"]
    estado["vida_atual"] = proporcao * estado["vida_maxima"]
    return estado

## 4. Motor de combate

In [ ]:
# Motor numérico compilado para tornar viável a enumeração completa.
# NumPy representa atributos em vetores; Numba compila o motor numerico para acelerar os duelos.
import numpy as np
from numba import njit

ATR = ('vida_maxima','regeneracao','poder_espiritual','bonus_dano_arma','bonus_cadencia',
       'bonus_carregador','resistencia_balistica','resistencia_espiritual','reducao_cooldown',
       'reducao_cooldown_infusao','roubo_vida_balistico','roubo_vida_espiritual',
       'amplificacao_cura','amplificacao_regeneracao')
# Mapeia nomes de efeitos de habilidade para codigos inteiros usados pelo motor compilado.
TIPOS = {n:i for i,n in enumerate(('direto','atrasado','barreira','continuo','canal','cubo',
                                   'cura_canal','cura_continua','incendiar','obsessao','voo','corvo','execucao',
                                   'bola','resiliencia','ritual'))}
# Ordem fixa das colunas que representam cada habilidade no vetor numerico.
COL_H = ('dano','escala_dano','cura','escala_cura','roubo_vida','alcance_m','duracao_s','atraso_s',
         'cooldown_s','acumulo_por_bala','max_acumulos','janela_acumulo_s','bonus_execucao',
         'escala_execucao','sangramento_pct_s','cargas','intervalo_cargas_s',
         'resistencia_ativa','regen_dano','regen_duracao_s')
# a: atributos (três regimes de vida); w: dano, cadência, clip, recarga, balas,
# spread, início/fim da queda, multiplicadores, alcance máximo, recarga por bala, atraso inicial.
@lru_cache(maxsize=30000)
# Precalcula tres conjuntos de atributos: vida acima de 65%, intermediaria e abaixo de 50%.
def _matriz_atributos(heroi, inv):
    return np.array([[a[k] for k in ATR] for a in (_atributos(heroi,inv,True,False),
                      _atributos(heroi,inv,False,False),_atributos(heroi,inv,False,True))],dtype=np.float64)


# Converte os atributos da arma e as habilidades do heroi em arrays de ponto flutuante.
def _preparar_combatente(heroi):
    b=config_herois[heroi]; ar=alcances_armas[heroi]
    w=np.array([b['damage_per_shot'],b['shots_per_second'],b['clip_size'],b['reload_duration'],b['bullets'],
        ar['spread'],ar['queda_inicio_m'],ar['queda_fim_m'],ar['escala_inicio'],ar['escala_fim'],ar['alcance_max_m'],
        float(ar['recarga_por_bala']),ar['recarga_inicial_s']],dtype=np.float64)
    h=np.array([[TIPOS[x['tipo']],*[x[k] for k in COL_H]] for x in habilidades_base[heroi]],dtype=np.float64)
    return w,h

ARMAS_HABILIDADES = {h:_preparar_combatente(h) for h in HEROIS}

@njit
# Chance de acerto aproximada em funcao da dispersao e distancia; nao e um dado medido no jogo.
def _chance(spread, distancia):
    # Hipótese comum a todos; não é uma taxa medida em partidas reais.
    return .90/(1.0+(distancia/35.0)**2*(1.0+spread)+(2./max(1.,distancia))**2/(1.+4.*spread))


def probabilidade_acerto(heroi, distancia):
    return float(_chance(alcances_armas[heroi]['spread'],float(distancia)))

@njit
# Calcula o multiplicador de dano pela distancia, com interpolacao linear na faixa de queda.
def _queda(w,d):
    if d>w[10]: return 0.
    if d<=w[6]: return w[8]
    if d>=w[7] or w[7]<=w[6]: return w[9]
    return w[8]+(d-w[6])/(w[7]-w[6])*(w[9]-w[8])

@njit
# Cada duelo comeca com municao cheia, sem barreira e com os temporizadores zerados.
def _estado_combatente(w,h,a):
    # arma: munição, capacidade, recarga restante, próximo disparo;
    # extras: barreira, duração, acúmulos, janela, queimadura.
    cap=round(w[2]*(1+a[5])); arma=np.array([cap,cap,0.,0.]); extras=np.zeros(6)
    # habilidades: cooldown, ativo, atraso, dano pendente, cargas, recarga de cargas, chance de aplicação.
    hs=np.zeros((len(h),7)); hs[:,4]=h[:,16]; hs[:,6]=1.
    return arma,extras,hs

@njit
# Detecta se a habilidade de voo esta ativa para ajustar a distancia efetiva.
def _voando(h,hs):
    for j in range(len(h)):
        if h[j,0]==10 and hs[j,1]>1e-9: return True
    return False

@njit
# Calcula um passo de tempo para um combatente: dano, cura, protecao e resistencia temporaria.
def _turno_num(w,h,arma,x,hs,a,vida,maxvida,alvo,maxalvo,d,dt):
    b=e=cura=roubo=0.; bloqueado=protegido=False
    x[1]=max(0.,x[1]-dt)
    if x[1]<=1e-9: x[0]=0.
    pa=.90/(1.+.4*(d/35.)**2) # chance estimada de aplicação das habilidades ofensivas
    # Avança relógios e resolve dano atrasado e disparos carregados.
    # Atualiza recargas e resolve efeitos que ficaram pendentes em passos anteriores.
    for j in range(len(h)):
        t=int(h[j,0]); s=hs[j]
        s[0]=max(0.,s[0]-dt)
        if t==12 and s[4]<h[j,16]:
            s[5]=max(0.,s[5]-dt)
            if s[5]<=1e-9:
                s[4]+=1
                if s[4]<h[j,16]: s[5]=h[j,9]*(1-a[8])
        if s[2]>1e-9:
            if t==12 or t==15: bloqueado=True
            s[2]=max(0.,s[2]-dt)
            if s[2]<=1e-9:
                # A execucao da Vindicta recebe bonus conforme a vida restante do alvo.
                if t==12:
                    bonus=max(0.,min(1.,1.-2.*alvo/maxalvo))
                    e+=(h[j,1]+h[j,2]*a[2]+bonus*(h[j,13]+h[j,14]*a[2]))*pa*d/(d+8.)
                elif t==15: s[1]=h[j,7]
                else: e+=s[3]
                s[3]=0.
        if s[1]>1e-9 and (t==4 or t==5 or t==6 or t==13 or t==15): bloqueado=True
    # Escolhe uma habilidade ativa disponivel e dentro do alcance; passivas sao tratadas depois.
    if not bloqueado:
        primeira=True
        for j in range(len(h)):
            t=int(h[j,0]); s=hs[j]; v=h[j]
            if t==8 or t==9 or t==14: continue
            infundida=primeira; primeira=False
            if s[0]>1e-9 or s[1]>1e-9 or s[2]>1e-9 or (v[6]>0 and d>v[6]): continue
            # Habilidades defensivas e de execucao so ativam nas condicoes de vida definidas no simulador.
            if (t==2 or t==5 or t==6 or t==7) and vida>.8*maxvida: continue
            if t==12 and (alvo>=.5*maxalvo or s[4]<1): continue
            if t==10 and d<8.: continue
            # Calcula reducao de cooldown, dano e cura com os bonus espirituais dos itens.
            red=min(.9,a[8]+(a[9] if infundida else 0.))
            s[0]=v[9]*(1-red)
            dano=v[1]+v[2]*a[2]; cv=v[3]+v[4]*a[2]
            if t==0: e+=dano*pa; cura+=cv
            elif t==1:
                # Dano atrasado e multiplicado pela chance de nao escapar; nao ha movimento real.
                s[2]=v[8]; s[3]=dano*pa*math.exp(-.25*v[8]) # chance de não escapar
            elif t==2: x[0]=cv; x[1]=v[7]
            elif t==15: s[2]=v[8]; bloqueado=True
            elif t==12:
                if s[4]==v[16]: s[5]=v[9]*(1-red)
                s[4]-=1; s[0]=v[17]; s[2]=v[8]; bloqueado=True
            else:
                s[1]=v[7]; s[6]=pa
                if t==11 or t==13: e+=dano*pa
            break
    voo=0.; passiva=-1; resistencia=0.
    # Percorre habilidades ativas para aplicar efeitos continuos, protecao, voo ou sangramento.
    for j in range(len(h)):
        t=int(h[j,0]); s=hs[j]; v=h[j]
        if t==8 or t==9: passiva=j
        if s[1]<=1e-9: continue
        tempo=min(dt,s[1]); s[1]=max(0.,s[1]-dt)
        if t==4 or t==5 or t==6 or t==13 or t==15: bloqueado=True
        if t==5: protegido=True
        if t==3 or t==4 or t==15:
            dano=(v[1]+v[2]*a[2])*tempo*pa; e+=dano; roubo+=dano*v[5]
        if t==5 or t==6 or t==7: cura+=(v[3]+v[4]*a[2])*tempo
        if t==10: voo=v[1]+v[2]*a[2]
        if t==13 or t==15: resistencia=max(resistencia,v[18])
        if t==11: e+=alvo*v[15]*tempo*s[6]
    x[3]=max(0.,x[3]-dt)
    if x[3]<=1e-9: x[2]=0.
    # Se nao estiver impedido por uma habilidade, atualiza recarga, disparos e efeitos por acerto.
    if not bloqueado:
        # Dano esperado: p(acerto) x dano se acertar; munição gasta mesmo no erro.
        chance=_chance(w[5],d); queda=_queda(w,d); shots=0
        if arma[2]>0.:
            arma[2]=max(0.,arma[2]-dt)
            if arma[2]==0.: arma[0]=arma[1]
        else:
            arma[3]-=dt
            while arma[3]<=0. and arma[0]>0.:
                arma[0]-=1; shots+=1; arma[3]+=1./(w[1]*(1+a[4]))
                if arma[0]==0.:
                    # Armas com recarga por bala usam tempo inicial mais capacidade vezes tempo por bala.
                    arma[2]=w[12]+arma[1]*w[3] if w[11]>0 else w[3]
                    arma[3]=0.; break
        extra=0.
        if passiva>=0 and h[passiva,0]==9: extra=h[passiva,1]*x[2]*w[4]
        # Dano balistico usa a queda por alcance e o valor esperado da chance de acerto.
        b=shots*(w[0]*(1+a[3])+extra)*queda*chance
        acertos=shots*w[4]*chance if queda>0 else 0.
        e+=acertos*voo
        # Acertos esperados alimentam as passivas de acumulacao e queimadura.
        if passiva>=0 and acertos>0:
            v=h[passiva]; x[3]=v[12]; x[2]=min(v[11],x[2]+acertos*v[10])
            if v[0]==8 and (x[2]>=v[11] or x[4]>1e-9): x[4]=v[7]; x[2]=0.
    if passiva>=0 and x[4]>1e-9:
        v=h[passiva]; tempo=min(dt,x[4]); e+=(v[1]+v[2]*a[2])*tempo; x[4]=max(0.,x[4]-dt)
    return b,e,cura,roubo,protegido,resistencia

@njit
# Resistencias reduzem o dano recebido; barreiras absorvem antes de reduzir a vida.
def _dano_num(t,x,a,protegido,vida,resistencia):
    if protegido: return 0.,0.,0.
    b=t[0]*(1-a[6])*(1-resistencia); e=t[1]*(1-a[7])*(1-resistencia); total=b+e
    if total<=0: return 0.,0.,0.
    absorvido=min(x[0],total); x[0]-=absorvido; f=min(vida,total-absorvido)/total
    return b*f,e*f,t[3]*(1-a[7])*(1-resistencia)*f

@njit
# Aplica dano, regeneracao e roubo de vida; a vida nunca ultrapassa a maxima.
def _vida_num(v,m,d,b,e,roubo,cura,a,dt):
    if d>=v: return 0.
    return min(m,v-d+a[1]*(1+a[13])*dt+(cura+roubo+b*a[10]+e*a[11])*(1+a[12]))

@njit
# Representa a regeneracao de uma passiva que recupera parte do dano recebido.
def _regen_passiva(h,x,dano,dt):
    for j in range(len(h)):
        if h[j,0]==14:
            x[5]+=dano*h[j,19]
            cura=x[5]*dt/h[j,20]; x[5]-=cura; return cura
    return 0.

@njit
# Simula os dois herois simultaneamente em passos de 0,1 s ate derrota ou limite de tempo.
def _duelo_num(a,ai,w,h,wi,hi,vida,vida_maxima,fracao_inimigo,distancia,dt,limite):
    max_i=ai[0,0]; vi=max_i*fracao_inimigo
    arma,x,hs=_estado_combatente(w,h,a[0]); armi,xi,hsi=_estado_combatente(wi,hi,ai[0]); tick=0
    for tick in range(1,math.ceil(limite/dt)+1):
        # Seleciona os atributos corretos conforme a vida restante, atualizada a cada passo.
        at=a[0] if vida>.65*vida_maxima else (a[2] if vida<.5*vida_maxima else a[1])
        ati=ai[0] if vi>.65*max_i else (ai[2] if vi<.5*max_i else ai[1])
        # Distância horizontal fixa. Voo acrescenta altitude aproximada de 6 m.
        altitude=6. if _voando(h,hs)!=_voando(hi,hsi) else 0.
        d=math.sqrt(distancia*distancia+altitude*altitude)
        # Calcula os ataques de ambos antes de atualizar suas vidas, permitindo derrotas simultaneas.
        t=_turno_num(w,h,arma,x,hs,at,vida,vida_maxima,vi,max_i,d,dt)
        ti=_turno_num(wi,hi,armi,xi,hsi,ati,vi,max_i,vida,vida_maxima,d,dt)
        b,e,roubo=_dano_num(t,xi,ati,ti[4],vi,ti[5]); bi,ei,ri=_dano_num(ti,x,at,t[4],vida,t[5])
        cura=t[2]+_regen_passiva(h,x,bi+ei,dt); curai=ti[2]+_regen_passiva(hi,xi,b+e,dt)
        vida,vi=(_vida_num(vida,vida_maxima,bi+ei,b,e,roubo,cura,at,dt),_vida_num(vi,max_i,b+e,bi,ei,ri,curai,ati,dt))
        if vida<=0 or vi<=0: break
    return vida,vi,tick*dt

## 5. Resultado do duelo e recompensa

In [ ]:
# Cache dos duelos identicos para nao repetir simulacoes caras durante o treinamento.
@lru_cache(maxsize=300000)
def _combate_em_cache(heroi, vida, vida_maxima, inventario, inimigo, distancia, fracao_inimigo, dt, limite):
    w,h=ARMAS_HABILIDADES[heroi]; wi,hi=ARMAS_HABILIDADES[inimigo]
    v,vi,tempo=_duelo_num(_matriz_atributos(heroi,inventario),_matriz_atributos(inimigo,()),w,h,wi,hi,
                         float(vida),float(vida_maxima),float(fracao_inimigo),float(distancia),float(dt),float(limite))
    # Uma vitoria exige sobreviver e zerar a vida do adversario; duplo KO ou tempo limite pode empatar.
    win=v>0 and vi==0; empate=not win and ((v==0 and vi==0) or (v>0 and vi>0))
    return {'vitoria':win,'empate':empate,'vida_restante':max(.01,round(v,2)) if v>0 else 0.,
            'vida_inimigo':max(.01,round(vi,2)) if vi>0 else 0.,'duracao':round(tempo,2),'distancia':distancia}


# Recompensa do RL: vitoria positiva com bonus de vida; derrota negativa com bonus de dano.
def recompensa_combate(resultado,vida_maxima,vida_max_inimigo):
    # Vitória recebe bônus de sobrevivência; derrota recebe bônus pelo dano líquido ao inimigo.
    if resultado['vitoria']:
        return round(1.+.25*resultado['vida_restante']/vida_maxima,6)
    return round(-1.+.25*(1.-resultado['vida_inimigo']/vida_max_inimigo),6)


# Recebe o estado e chama o motor de combate; o inimigo inicia com vida cheia e sem itens.
def simular_combate(estado, adversario=None, dt=DT):
    inimigo = adversario if adversario is not None else estado["adversario"]
    if inimigo not in HEROIS or inimigo == estado["heroi"]:
        raise ValueError("O adversário deve ser outro herói do conjunto treinado.")
    if not 0 < FRACAO_VIDA_INIMIGO <= 1 or not 0 < dt <= TEMPO_MAX_COMBATE:
        raise ValueError("Configuração de combate inválida.")
    return _combate_em_cache(estado["heroi"], estado["vida_atual"], estado["vida_maxima"],
                             tuple(sorted(estado["inventario"])), inimigo,
                             estado["distancia"], FRACAO_VIDA_INIMIGO, dt, TEMPO_MAX_COMBATE).copy()

## 6. Estados, transições e representação do Q-learning

In [ ]:
def sortear_adversario(heroi, rng=None):
    return (rng or random).choice([h for h in HEROIS if h != heroi])


# reset cria o estado inicial padrao; adversario e distancia podem ser definidos ou sorteados.
def reset(heroi="Infernus", adversario=None, rng=None, distancia=None):
    if heroi not in HEROIS:
        raise ValueError("Herói não disponível.")
    inimigo = adversario if adversario is not None else sortear_adversario(heroi, rng)
    if inimigo not in HEROIS or inimigo == heroi:
        raise ValueError("Escolha outro herói como adversário.")
    distancia = (rng or random).choice(DISTANCIAS) if distancia is None else distancia
    if distancia not in DISTANCIAS:
        raise ValueError("Selecione uma distância de 2, 15 ou 25 metros.")
    vida = float(config_herois[heroi]["vida_base"])
    return {"heroi": heroi, "etapas_restantes": ETAPAS, "almas": ALMAS_INICIAIS,
            "vida_atual": vida, "vida_maxima": vida, "inventario": [],
            "adversario": inimigo, "distancia": distancia, "terminado": False}


# Discretiza a vida nas quatro faixas representativas; isso reduz a Tabela Q.
def faixa_vida(vida, maxima):
    """Aproxima ao percentual representativo mais próximo; empate vai para cima."""
    return min(VIDAS_PCT, key=lambda p: (abs(p - 100 * vida / maxima), -p))


# Cria estados de consulta com saldo, vida e inventario independentes de vitorias passadas.
def estado_cenario(heroi, adversario, distancia, etapas, almas, vida_pct, inventario=()):
    """Estado independente do histórico de vitórias. Usado também nos sucessores."""
    estado = reset(heroi, adversario, distancia=distancia)
    if etapas not in range(1, ETAPAS + 1) or almas not in SALDOS or vida_pct not in VIDAS_PCT:
        raise ValueError("Etapas, saldo ou percentual de vida fora das opções.")
    if len(set(inventario)) != len(inventario) or any(a not in custos_itens for a in inventario):
        raise ValueError("Inventário inválido.")
    maxima = calcular_atributos(inventario, heroi=heroi)["vida_maxima"]
    estado.update(etapas_restantes=etapas, almas=almas, inventario=list(sorted(inventario)),
                  vida_maxima=maxima, vida_atual=maxima * vida_pct / 100)
    return estado


# Reusa o resultado do duelo para o mesmo inventario, sem depender de saldo ou duelos restantes.
@lru_cache(maxsize=500000)
def resultado_inventario(heroi, vida_pct, inventario, inimigo, distancia):
    """O combate depende do inventário após compra, não do saldo ou horizonte."""
    estado = estado_cenario(heroi, inimigo, distancia, 1, 0, vida_pct, inventario)
    resultado = simular_combate(estado)
    if not resultado["vitoria"]:
        return recompensa_combate(resultado, estado["vida_maxima"], float(config_herois[inimigo]["vida_base"])), 0
    vida = resultado["vida_restante"]
    a = calcular_atributos(inventario, vida / estado["vida_maxima"], heroi)
    # Em caso de vitoria, soma a regeneracao de 45 s e aproxima a vida ao percentual mais proximo.
    regen = (a["regeneracao"] + a["regeneracao_fora_combate"]) * (1 + a["amplificacao_regeneracao"])
    recuperada = min(estado["vida_maxima"], vida + regen * TEMPO_RECUPERACAO)
    return recompensa_combate(resultado, estado["vida_maxima"], float(config_herois[inimigo]["vida_base"])), faixa_vida(recuperada, estado["vida_maxima"])


# Executa uma compra ou guarda almas, simula o duelo e calcula recompensa e proximo estado.
def transicao_compra(estado, acao):
    """Compra, duelo e recuperação; discretiza a vida se continuar."""
    pct = faixa_vida(estado["vida_atual"], estado["vida_maxima"])
    novo = atualizar_atributos(executar_acao(estado, acao))
    inventario = tuple(sorted(novo["inventario"]))
    novo["vida_atual"] = novo["vida_maxima"] * pct / 100
    r, proxima_pct = resultado_inventario(novo["heroi"], pct, inventario, novo["adversario"], novo["distancia"])
    novo["etapas_restantes"] -= 1
    # O episodio termina ao perder ou ao completar o ultimo duelo.
    terminado = r < 0 or novo["etapas_restantes"] == 0
    novo["terminado"] = terminado
    # Se continuar, adiciona almas ate o teto e usa a vida recuperada e discretizada.
    if not terminado:
        novo["almas"] = min(SALDOS[-1], novo["almas"] + ALMAS_POR_VITORIA)
        novo["vida_atual"] = novo["vida_maxima"] * proxima_pct / 100
    else:
        novo["vida_atual"] = simular_combate(novo)["vida_restante"]
    return novo, r, terminado


def contextos(heroi):
    return [(inimigo, distancia) for inimigo in HEROIS if inimigo != heroi for distancia in DISTANCIAS]


# step e a interface com Q-learning: devolve novo estado, recompensa e indicador de termino.
def step(estado, acao, rng=None):
    novo, recompensa, terminado = transicao_compra(estado, acao)
    if not terminado:
        novo["adversario"], novo["distancia"] = (rng or random).choice(contextos(novo["heroi"]))
    return novo, recompensa, terminado


# A chave imutavel e usada como identificador na Tabela Q; inventario ordenado evita duplicatas.
def chave_estado(estado):
    return (estado["heroi"], estado["etapas_restantes"], estado["almas"],
            faixa_vida(estado["vida_atual"], estado["vida_maxima"]), tuple(sorted(estado["inventario"])),
            estado["adversario"], estado["distancia"])


# Reconstroi um estado a partir da chave usada no treinamento e no chatbot.
def estado_da_chave(chave):
    heroi, etapas, almas, pct, inventario, inimigo, distancia = chave
    return estado_cenario(heroi, inimigo, distancia, etapas, almas, pct, inventario)


if __name__ == "__main__":
    print(f"{len(HEROIS)} heróis, {len(itens)} itens e {len(HEROIS) * (len(HEROIS) - 1)} duelos possíveis.")
    estado = reset("Infernus", "Abrams", distancia=15)
    print("Estado:", estado)
    print("Combate sem compras:", simular_combate(estado))
    print("Ações disponíveis:", [nome_acao(a) for a in acoes_validas(estado)])

## 7. Conferir os atributos e os efeitos dos itens

In [ ]:
# Examina as tabelas e os efeitos interpretados pelo ambiente.
from IPython.display import display
import json

# Exibe os atributos efetivamente selecionados para os cinco herois.
display(heroes[["nome_heroi", "vida_base", "regeneracao_base", "damage_per_shot", "shots_per_second", "clip_size", "reload_duration"]])

# Exibe os 13 itens filtrados do catalogo.
display(itens[["nome", "class_name", "item_id", "custo"]])

# Exemplo: o CSV guarda um objeto JSON na coluna properties_json.
# O item de vida e selecionado pelo identificador interno, nao pelo nome traduzido.
linha = itens.loc[itens["class_name"] == "upgrade_health"].iloc[0]
propriedades = json.loads(linha["properties_json"])
print("Item:", linha["nome"])
print("BonusHealth na celula JSON:", propriedades["BonusHealth"]["value"])
print("Efeito interpretado pelo ambiente:", efeitos_itens[int(linha["item_id"])])

# Demonstra o efeito de equipar o item no Infernus.
print("Vida sem item:", calcular_atributos([], heroi="Infernus")["vida_maxima"])
print("Vida com item:", calcular_atributos([int(linha["item_id"])], heroi="Infernus")["vida_maxima"])

# Mostra tambem os outros dois CSVs carregados no ambiente.
display(tabela_alcances)
display(tabela_habilidades[["heroi", "slot", "tipo", "dano", "alcance_m", "cooldown_s"]])

## 8. Testar uma transição

In [ ]:
print("Heróis:", HEROIS)
display(itens[["nome", "custo"]])
# Seleciona dois itens para demonstrar um estado independente do historico de vitorias.
inv = list(custos_itens)[:2]
estado = estado_cenario("Infernus", "Abrams", 25, 4, 3200, 25, inv)
print("Estado:", estado)
print("Ações:", [nome_acao(a) for a in acoes_validas(estado)])
# Executa a acao de guardar almas e observa a transicao do ambiente.
novo, recompensa, terminou = step(estado, ACAO_GUARDAR)
print("Transição:", novo, recompensa, terminou)
# Compara a chance base de acerto aproximada em cada distancia.
print("Chance base de acerto por distância:")
for h in HEROIS:
    print(h, {d: round(probabilidade_acerto(h,d),3) for d in DISTANCIAS})